# 03c: Logit Lens & Component Attribution: Circuit Analysis

## Overview
Analyzes logit lens P(correct) trajectories and component attribution (attention vs MLP)
under **mean ablation** (circuit mode). Non-circuit edges have their outputs replaced
with dataset-mean activations, isolating the computational path through the ACDC-discovered
circuit.

## Key Questions
1. How does ablation change the P(correct) trajectory? Does the circuit preserve the
   layer-wise build-up of the correct prediction?
2. Does the convergence layer shift under circuit ablation?
3. How does the attention vs MLP attribution profile change when only circuit edges are active?
4. Which bands and models lose the most logit-lens structure?

## Hypothesis Domain: R2b-circuit
- Circuit should preserve convergence ordering across bands (low converges later)
- Circuit may shift convergence layer later if non-circuit components contributed early
- Attribution balance (attn vs MLP) may shift if the circuit is attn-dominated or MLP-dominated

## Sections
1. Circuit Logit Lens P(correct) Trajectory
2. Circuit Convergence Layer
3. Circuit Component Attribution (Attention vs MLP)
4. Base vs Circuit Comparison
5. Cross-Model Patterns

## Data Sources
- Base activations: `outputs/extraction/activations/`
- Circuit activations: `outputs/extraction/circuit_activations/`
- Base logit lens analysis: `outputs/logit_lens/base/analysis/`

In [1]:
import sys
import numpy as np
import pandas as pd
from pathlib import Path
from functools import partial as _partial

sys.path.insert(0, str(Path.cwd()))

from utils.constants import (
    MODELS,
    BANDS,
    DRAWS,
    FREQUENCY_RANK,
    MODEL_INFO,
    BAND_COLORS,
    BAND_NAMES,
    MODEL_COLORS,
    MODEL_CAPACITY,
    MODEL_D_MODEL,
    ACTIVATIONS_DIR,
    RANDOM_SEED,
    CONVERGENCE_THRESHOLD,
    get_domain_dirs,
)
from utils.data_loading import (
    load_extracted_activations,
    save_analysis,
    build_representational_df,
    load_domain_csv,
)
from utils.circuit_loading import (
    load_circuit_activations,
    load_base_and_circuit,
    load_prune_scores,
    get_circuit_mask,
    get_circuit_summary,
)
from utils.plotting import setup_plotting, save_figure, plot_logit_lens_heatmap

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

setup_plotting()

CIRCUIT_ANALYSIS, CIRCUIT_VIZ = get_domain_dirs("logit_lens", "circuit")
COMP_ANALYSIS, COMP_VIZ = get_domain_dirs("logit_lens", "comparison")
save_analysis_circuit = _partial(save_analysis, analysis_dir=CIRCUIT_ANALYSIS)
save_figure_circuit = _partial(save_figure, viz_dir=CIRCUIT_VIZ)
save_analysis_comp = _partial(save_analysis, analysis_dir=COMP_ANALYSIS)
save_figure_comp = _partial(save_figure, viz_dir=COMP_VIZ)

print(f"Models: {MODELS}")
print(f"Bands: {BANDS}")
print(f"Draws: {DRAWS}")
print(f"Convergence threshold: {CONVERGENCE_THRESHOLD}")
print(f"Circuit analysis: {CIRCUIT_ANALYSIS}")
print(f"Comparison analysis: {COMP_ANALYSIS}")

Models: ['pythia-70m', 'pythia-160m', 'pythia-410m', 'pythia-1b', 'pythia-1.4b']
Bands: ['low', 'medium', 'high', 'very_high', 'control']
Draws: ['draw_1', 'draw_2', 'draw_3']
Convergence threshold: 0.9
Circuit analysis: LSC_circuit_analysis/03_Phase_Representational/outputs/logit_lens/circuit/analysis
Comparison analysis: LSC_circuit_analysis/03_Phase_Representational/outputs/logit_lens/comparison/analysis


## 1. Circuit Logit Lens P(correct) Trajectory

Load circuit-mode activations and extract the pre-computed `logit_lens_prob_correct`
per model/band/draw/layer. This measures how the probability of the correct target
token builds up through layers when only circuit edges are active.

In [2]:
rows_prob = []

for model in MODELS:
    n_layers = MODEL_INFO[model]["n_layers"]
    for draw in DRAWS:
        for band in BANDS:
            try:
                data = load_circuit_activations(model, band, draw)
            except FileNotFoundError:
                continue

            prob = data["logit_lens_prob_correct"]  # (N, n_layers)
            mean_prob = prob.mean(axis=0)
            std_prob = prob.std(axis=0)
            median_prob = np.median(prob, axis=0)

            for layer in range(n_layers):
                rows_prob.append(
                    {
                        "model": model,
                        "draw": draw,
                        "band": band,
                        "layer": layer,
                        "mean_prob_correct": float(mean_prob[layer]),
                        "std_prob_correct": float(std_prob[layer]),
                        "median_prob_correct": float(median_prob[layer]),
                    }
                )

df_circuit_prob = pd.DataFrame(rows_prob)
save_analysis_circuit(df_circuit_prob, "03c_circuit_prob_correct.csv")
print(f"Circuit P(correct) trajectory: {len(df_circuit_prob)} rows")
df_circuit_prob.head()

Circuit P(correct) trajectory: 1230 rows


,model,draw,band,layer,mean_prob_correct,std_prob_correct,median_prob_correct
0,pythia-70m,draw_1,low,0,7.991129e-10,9.605091e-09,2.378680e-16
1,pythia-70m,draw_1,low,1,5.494817e-08,8.138901e-07,5.885292e-16
2,pythia-70m,draw_1,low,2,8.381735e-07,1.131239e-05,2.040144e-14
3,pythia-70m,draw_1,low,3,4.509212e-02,1.858030e-01,9.930671e-09
4,pythia-70m,draw_1,low,4,1.451562e-01,3.189201e-01,6.987067e-06


In [3]:
# Visualization: P(correct) trajectory per model, colored by band
for model in MODELS:
    df_m = df_circuit_prob[
        (df_circuit_prob["model"] == model) & (df_circuit_prob["draw"] == "draw_1")
    ]
    if df_m.empty:
        continue

    fig, ax = plt.subplots(figsize=(12, 6))
    for band in BANDS:
        bd = df_m[df_m["band"] == band].sort_values("layer")
        if bd.empty:
            continue
        color = BAND_COLORS.get(band, "gray")
        label = BAND_NAMES.get(band, band)
        ax.plot(
            bd["layer"],
            bd["mean_prob_correct"],
            color=color,
            label=label,
            marker="o",
            markersize=3,
        )
        ax.fill_between(
            bd["layer"],
            bd["mean_prob_correct"] - bd["std_prob_correct"],
            bd["mean_prob_correct"] + bd["std_prob_correct"],
            color=color,
            alpha=0.12,
        )

    ax.set_xlabel("Layer")
    ax.set_ylabel("Mean P(correct)")
    ax.set_title(f"Circuit Logit Lens: P(correct) Trajectory \u2014 {model}")
    ax.legend(fontsize=8)
    ax.set_ylim(bottom=0)
    fig.tight_layout()
    save_figure_circuit(fig, f"viz_03c_01_circuit_prob_trajectory_{model}.png")

In [4]:
# Heatmap: bands x layers for each model (P(correct) averaged across draws)
for model in MODELS:
    n_layers = MODEL_INFO[model]["n_layers"]
    model_data = df_circuit_prob[df_circuit_prob["model"] == model]
    if model_data.empty:
        continue

    avg = (
        model_data.groupby(["layer", "band"])["mean_prob_correct"].mean().reset_index()
    )
    pivot = avg.pivot(index="band", columns="layer", values="mean_prob_correct")
    pivot = pivot.reindex(index=[b for b in BANDS if b in pivot.index])
    pivot = pivot.reindex(columns=list(range(n_layers)))

    bands_present = [b for b in BANDS if b in pivot.index]
    layers = list(range(n_layers))
    values = pivot.values

    fig = plot_logit_lens_heatmap(
        values,
        bands_present,
        layers,
        title=f"Circuit P(correct) by Layer \u2014 {model}",
        cmap="YlOrRd",
        fmt=".2f",
        xlabel="Layer",
        ylabel="Band",
    )
    save_figure_circuit(fig, f"viz_03c_02_circuit_prob_heatmap_{model}.png")

## 2. Circuit Convergence Layer

Find the layer where P(correct) first exceeds `CONVERGENCE_THRESHOLD` \u00d7 final P(correct)
under circuit ablation. Compare convergence timing across bands.

In [5]:
from utils.logit_lens import compute_convergence_layer

rows_conv = []

for model in MODELS:
    n_layers = MODEL_INFO[model]["n_layers"]
    for draw in DRAWS:
        for band in BANDS:
            try:
                data = load_circuit_activations(model, band, draw)
            except FileNotFoundError:
                continue

            prob = data["logit_lens_prob_correct"]  # (N, n_layers)
            conv_layers = compute_convergence_layer(
                prob, threshold=CONVERGENCE_THRESHOLD
            )  # (N,)

            converged_mask = conv_layers < n_layers
            n_converged = int(converged_mask.sum())
            n_total = len(conv_layers)

            if n_converged > 0:
                conv_vals = conv_layers[converged_mask]
                mean_conv = float(conv_vals.mean())
                median_conv = float(np.median(conv_vals))
                std_conv = float(conv_vals.std())
            else:
                mean_conv = float(n_layers)
                median_conv = float(n_layers)
                std_conv = 0.0

            rows_conv.append(
                {
                    "model": model,
                    "draw": draw,
                    "band": band,
                    "mean_convergence_layer": mean_conv,
                    "median_convergence_layer": median_conv,
                    "std_convergence_layer": std_conv,
                    "frac_converged": float(n_converged / n_total),
                    "n_converged": n_converged,
                    "n_total": n_total,
                    "n_layers": n_layers,
                    "frac_convergence_layer": mean_conv / n_layers,
                }
            )

df_circuit_conv = pd.DataFrame(rows_conv)
save_analysis_circuit(df_circuit_conv, "03c_circuit_convergence.csv")
print(f"Circuit convergence: {len(df_circuit_conv)} rows")
if not df_circuit_conv.empty:
    print(f"\nMean circuit convergence layer by model x band:")
    print(
        df_circuit_conv.groupby(["model", "band"])["mean_convergence_layer"]
        .mean()
        .unstack("band")
        .round(2)
    )

Circuit convergence: 75 rows

Mean circuit convergence layer by model x band:
band         control   high    low  medium  very_high
model                                                
pythia-1.4b    15.03  16.57  21.45   19.39      14.50
pythia-160m     8.69   8.97   9.55    9.21       8.65
pythia-1b      11.05  11.35  12.40   11.76      11.01
pythia-410m    17.71  18.68  21.04   20.08      17.41
pythia-70m      4.35   4.49   4.72    4.63       4.35


In [6]:
# Heatmap: mean circuit convergence layer (model x band)
if not df_circuit_conv.empty:
    conv_pivot = (
        df_circuit_conv.groupby(["model", "band"])["mean_convergence_layer"]
        .mean()
        .reset_index()
    )
    conv_wide = conv_pivot.pivot(
        index="model", columns="band", values="mean_convergence_layer"
    )
    conv_wide = conv_wide.reindex(
        index=MODELS, columns=[b for b in BANDS if b in conv_wide.columns]
    )

    fig, ax = plt.subplots(figsize=(10, 5))
    sns.heatmap(
        conv_wide,
        annot=True,
        fmt=".1f",
        cmap="YlOrRd",
        square=True,
        linewidths=0,
        linecolor="none",
        ax=ax,
    )
    ax.set_title(f"Circuit Convergence Layer (threshold={CONVERGENCE_THRESHOLD})")
    ax.set_xlabel("Band")
    ax.set_ylabel("Model")
    fig.tight_layout()
    save_figure_circuit(fig, "viz_03c_03_circuit_convergence_heatmap.png")

## 3. Circuit Component Attribution

Decompose the correct-token logit into attention and MLP contributions at each layer
under circuit ablation. Uses pre-extracted `attn_out_predpos` and `mlp_out_predpos`
from the circuit activations.

Attribution via dot product (no ln_final):
- `attn_contrib[l] = attn_out[l] \u00b7 W_U[:, target_id]`
- `mlp_contrib[l]  = mlp_out[l]  \u00b7 W_U[:, target_id]`

This requires the model's unembedding matrix `W_U`. We attempt to load the model;
if unavailable, we fall back to a norm-based proxy.

In [7]:
# Attempt to load W_U from models for proper attribution
USE_WU_ATTRIBUTION = False
model_W_U = {}  # model -> W_U numpy array (d_model, d_vocab)

try:
    from utils.extraction import load_model
    import gc

    for model_name in MODELS:
        try:
            print(f"Loading {model_name} for W_U extraction...")
            model_obj = load_model(model_name, device="cpu", verbose=False)
            W_U = model_obj.W_U.detach().cpu().numpy()  # (d_model, d_vocab)
            model_W_U[model_name] = W_U
            print(f"  W_U shape: {W_U.shape}")
            del model_obj
            gc.collect()
        except Exception as e:
            print(f"  Failed to load {model_name}: {e}")

    if len(model_W_U) > 0:
        USE_WU_ATTRIBUTION = True
        print(
            f"\nLoaded {len(model_W_U)}/{len(MODELS)} models. "
            f"Using W_U-based attribution."
        )
    else:
        print(f"\nNo models loaded. Falling back to norm-based proxy.")

except ImportError:
    print("transformer_lens not available. Using norm-based proxy.")
except Exception as e:
    print(f"Model loading failed: {e}. Using norm-based proxy.")

print(f"\nUSE_WU_ATTRIBUTION = {USE_WU_ATTRIBUTION}")

Loading pythia-70m for W_U extraction...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

Loaded pretrained model EleutherAI/pythia-70m into HookedTransformer
  W_U shape: (512, 50304)
Loading pythia-160m for W_U extraction...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

...

Loaded pretrained model EleutherAI/pythia-1b into HookedTransformer
  W_U shape: (2048, 50304)


Loading pythia-1.4b for W_U extraction...


Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

Loaded pretrained model EleutherAI/pythia-1.4b into HookedTransformer
  W_U shape: (2048, 50304)



Loaded 5/5 models. Using W_U-based attribution.

USE_WU_ATTRIBUTION = True


In [8]:
from utils.logit_lens import compute_component_attribution

rows_attrib = []

for model in MODELS:
    n_layers = MODEL_INFO[model]["n_layers"]
    W_U = model_W_U.get(model)

    for draw in DRAWS:
        for band in BANDS:
            try:
                data = load_circuit_activations(model, band, draw)
            except FileNotFoundError:
                continue

            attn = data["attn_out_predpos"]  # (N, n_layers, d_model)
            mlp = data["mlp_out_predpos"]  # (N, n_layers, d_model)
            tids = data["target_ids"]  # (N,)

            if USE_WU_ATTRIBUTION and W_U is not None:
                result = compute_component_attribution(attn, mlp, W_U, tids)
                attn_logit = result["attn_logit"]  # (N, n_layers)
                mlp_logit = result["mlp_logit"]  # (N, n_layers)
                attn_frac = result["attn_frac"]  # (N, n_layers)
                mlp_frac = result["mlp_frac"]  # (N, n_layers)
                method = "W_U"
            else:
                # Norm-based proxy
                attn_norms = np.linalg.norm(attn, axis=-1)
                mlp_norms = np.linalg.norm(mlp, axis=-1)
                total_norms = attn_norms + mlp_norms + 1e-10
                attn_frac = attn_norms / total_norms
                mlp_frac = mlp_norms / total_norms
                attn_logit = attn_norms
                mlp_logit = mlp_norms
                method = "norm_proxy"

            for layer in range(n_layers):
                rows_attrib.append(
                    {
                        "model": model,
                        "draw": draw,
                        "band": band,
                        "layer": layer,
                        "attn_logit_mean": float(attn_logit[:, layer].mean()),
                        "mlp_logit_mean": float(mlp_logit[:, layer].mean()),
                        "attn_frac": float(attn_frac[:, layer].mean()),
                        "mlp_frac": float(mlp_frac[:, layer].mean()),
                        "attn_logit_std": float(attn_logit[:, layer].std()),
                        "mlp_logit_std": float(mlp_logit[:, layer].std()),
                        "method": method,
                    }
                )

df_circuit_attrib = pd.DataFrame(rows_attrib)
save_analysis_circuit(df_circuit_attrib, "03c_circuit_attribution.csv")
print(f"Circuit attribution: {len(df_circuit_attrib)} rows")
if not df_circuit_attrib.empty:
    print(f"Method: {df_circuit_attrib['method'].unique()}")
    print(f"\nCircuit final-layer mean attn fraction (draw_1):")
    final = (
        df_circuit_attrib[df_circuit_attrib["draw"] == "draw_1"]
        .groupby(["model", "band"])
        .apply(lambda g: g.loc[g["layer"].idxmax(), "attn_frac"])
        .unstack("band")
    )
    print(final.round(3))

Circuit attribution: 1230 rows
Method: <ArrowStringArray>
['W_U']
Length: 1, dtype: str

Circuit final-layer mean attn fraction (draw_1):
band         control   high    low  medium  very_high
model                                                
pythia-1.4b    0.207  0.166  0.138   0.134      0.214
pythia-160m    0.190  0.190  0.197   0.193      0.188
pythia-1b      0.386  0.417  0.358   0.355      0.438
pythia-410m    0.166  0.144  0.138   0.136      0.166
pythia-70m     0.333  0.341  0.350   0.351      0.331


In [9]:
# Signed logit contribution trajectories under circuit ablation
for model in MODELS:
    model_data = df_circuit_attrib[
        (df_circuit_attrib["model"] == model) & (df_circuit_attrib["draw"] == "draw_1")
    ]
    if model_data.empty:
        continue

    bands_present = [b for b in BANDS if b in model_data["band"].unique()]
    n_bands = len(bands_present)
    fig, axes = plt.subplots(1, n_bands, figsize=(4 * n_bands, 5), sharey=True)
    if n_bands == 1:
        axes = [axes]

    for ax, band in zip(axes, bands_present):
        bd = model_data[model_data["band"] == band].sort_values("layer")
        layers = bd["layer"].values

        ax.plot(
            layers,
            bd["attn_logit_mean"],
            color="#1f77b4",
            label="Attention",
            marker="o",
            markersize=3,
        )
        ax.fill_between(
            layers,
            bd["attn_logit_mean"] - bd["attn_logit_std"],
            bd["attn_logit_mean"] + bd["attn_logit_std"],
            color="#1f77b4",
            alpha=0.12,
        )

        ax.plot(
            layers,
            bd["mlp_logit_mean"],
            color="#ff7f0e",
            label="MLP",
            marker="s",
            markersize=3,
        )
        ax.fill_between(
            layers,
            bd["mlp_logit_mean"] - bd["mlp_logit_std"],
            bd["mlp_logit_mean"] + bd["mlp_logit_std"],
            color="#ff7f0e",
            alpha=0.12,
        )

        ax.axhline(y=0, color="gray", linestyle="--", alpha=0.3)
        ax.set_xlabel("Layer")
        ax.set_title(BAND_NAMES.get(band, band))

    axes[0].set_ylabel("Logit Contribution to Correct Token")
    axes[-1].legend(fontsize=8)
    fig.suptitle(f"Circuit Signed Attribution \u2014 {model}", y=1.02)
    fig.tight_layout()
    save_figure_circuit(fig, f"viz_03c_04_circuit_attribution_signed_{model}.png")

In [10]:
# Heatmap: circuit attention fraction (model x band, averaged over layers and draws)
if not df_circuit_attrib.empty:
    attrib_summary = (
        df_circuit_attrib.groupby(["model", "band"])
        .agg(
            {
                "attn_frac": "mean",
                "mlp_frac": "mean",
            }
        )
        .reset_index()
    )

    for metric, title, cmap in [
        ("attn_frac", "Circuit: Mean Attention Attribution Fraction", "Blues"),
        ("mlp_frac", "Circuit: Mean MLP Attribution Fraction", "Oranges"),
    ]:
        pivot = attrib_summary.pivot(index="model", columns="band", values=metric)
        pivot = pivot.reindex(
            index=MODELS, columns=[b for b in BANDS if b in pivot.columns]
        )

        fig, ax = plt.subplots(figsize=(10, 5))
        sns.heatmap(
            pivot,
            annot=True,
            fmt=".3f",
            cmap=cmap,
            square=True,
            linewidths=0,
            linecolor="none",
            ax=ax,
        )
        ax.set_title(title)
        ax.set_xlabel("Band")
        ax.set_ylabel("Model")
        fig.tight_layout()
        suffix = metric.replace("_frac", "")
        save_figure_circuit(fig, f"viz_03c_05_circuit_attribution_{suffix}_heatmap.png")

## 4. Base vs Circuit Comparison

Compare base-model and circuit logit lens metrics:
- P(correct) trajectory overlay
- Convergence layer shift (circuit - base)
- Attribution profile differences

In [11]:
# Load base logit lens CSVs for comparison
try:
    df_base_prob = load_domain_csv(
        "logit_lens", "base", "03_prob_correct_trajectory.csv"
    )
    print(f"Base P(correct) trajectory: {len(df_base_prob)} rows")
except FileNotFoundError:
    print("Base P(correct) trajectory not found, skipping")
    df_base_prob = pd.DataFrame()

try:
    df_base_conv = load_domain_csv("logit_lens", "base", "03_convergence_layers.csv")
    print(f"Base convergence: {len(df_base_conv)} rows")
except FileNotFoundError:
    print("Base convergence not found, skipping")
    df_base_conv = pd.DataFrame()

try:
    df_base_attrib = load_domain_csv(
        "logit_lens", "base", "03_component_attribution.csv"
    )
    print(f"Base attribution: {len(df_base_attrib)} rows")
except FileNotFoundError:
    print("Base attribution not found, skipping")
    df_base_attrib = pd.DataFrame()

Base P(correct) trajectory: 1230 rows
Base convergence: 75 rows
Base attribution: 1230 rows


In [12]:
# P(correct) trajectory overlay: base (solid) vs circuit (dashed)
if not df_base_prob.empty and not df_circuit_prob.empty:
    for model in MODELS:
        fig, ax = plt.subplots(figsize=(12, 6))

        for band in BANDS:
            color = BAND_COLORS.get(band, "gray")
            label_base = BAND_NAMES.get(band, band)

            # Base (solid)
            base_bd = df_base_prob[
                (df_base_prob["model"] == model)
                & (df_base_prob["band"] == band)
                & (df_base_prob["draw"] == "draw_1")
            ].sort_values("layer")
            if not base_bd.empty:
                ax.plot(
                    base_bd["layer"],
                    base_bd["mean_prob_correct"],
                    color=color,
                    label=f"{label_base} (base)",
                    linewidth=2,
                    marker="o",
                    markersize=3,
                )

            # Circuit (dashed)
            circ_bd = df_circuit_prob[
                (df_circuit_prob["model"] == model)
                & (df_circuit_prob["band"] == band)
                & (df_circuit_prob["draw"] == "draw_1")
            ].sort_values("layer")
            if not circ_bd.empty:
                ax.plot(
                    circ_bd["layer"],
                    circ_bd["mean_prob_correct"],
                    color=color,
                    label=f"{label_base} (circuit)",
                    linewidth=2,
                    linestyle="--",
                    marker="s",
                    markersize=3,
                )

        ax.set_xlabel("Layer")
        ax.set_ylabel("Mean P(correct)")
        ax.set_title(f"P(correct) Trajectory: Base vs Circuit \u2014 {model}")
        ax.legend(fontsize=7, ncol=2)
        ax.set_ylim(bottom=0)
        fig.tight_layout()
        save_figure_comp(fig, f"viz_03c_06_prob_overlay_{model}.png")

In [13]:
# Convergence layer shift: circuit - base per band
if not df_base_conv.empty and not df_circuit_conv.empty:
    rows_shift = []

    for model in MODELS:
        for band in BANDS:
            for draw in DRAWS:
                base_row = df_base_conv[
                    (df_base_conv["model"] == model)
                    & (df_base_conv["band"] == band)
                    & (df_base_conv["draw"] == draw)
                ]
                circ_row = df_circuit_conv[
                    (df_circuit_conv["model"] == model)
                    & (df_circuit_conv["band"] == band)
                    & (df_circuit_conv["draw"] == draw)
                ]
                if base_row.empty or circ_row.empty:
                    continue

                base_val = base_row.iloc[0]["mean_convergence_layer"]
                circ_val = circ_row.iloc[0]["mean_convergence_layer"]
                n_layers = circ_row.iloc[0]["n_layers"]

                rows_shift.append(
                    {
                        "model": model,
                        "band": band,
                        "draw": draw,
                        "base_convergence": base_val,
                        "circuit_convergence": circ_val,
                        "convergence_shift": circ_val - base_val,
                        "frac_shift": (circ_val - base_val) / n_layers,
                    }
                )

    df_conv_shift = pd.DataFrame(rows_shift)
    save_analysis_comp(df_conv_shift, "03c_convergence_shift.csv")
    print(f"Convergence shift: {len(df_conv_shift)} rows")

    if not df_conv_shift.empty:
        print(f"\nMean convergence shift (circuit - base) by model x band:")
        print(
            df_conv_shift.groupby(["model", "band"])["convergence_shift"]
            .mean()
            .unstack("band")
            .round(2)
        )

        # Heatmap of convergence shift
        shift_pivot = (
            df_conv_shift.groupby(["model", "band"])["convergence_shift"]
            .mean()
            .reset_index()
        )
        shift_wide = shift_pivot.pivot(
            index="model", columns="band", values="convergence_shift"
        )
        shift_wide = shift_wide.reindex(
            index=MODELS, columns=[b for b in BANDS if b in shift_wide.columns]
        )

        # Use diverging colormap centered at 0
        vmax = max(abs(shift_wide.min().min()), abs(shift_wide.max().max()), 0.1)
        fig, ax = plt.subplots(figsize=(10, 5))
        sns.heatmap(
            shift_wide,
            annot=True,
            fmt=".2f",
            cmap="RdBu_r",
            square=True,
            linewidths=0,
            linecolor="none",
            vmin=-vmax,
            vmax=vmax,
            center=0,
            ax=ax,
        )
        ax.set_title("Convergence Layer Shift (Circuit - Base)")
        ax.set_xlabel("Band")
        ax.set_ylabel("Model")
        fig.tight_layout()
        save_figure_comp(fig, "viz_03c_07_convergence_shift_heatmap.png")

Convergence shift: 75 rows

Mean convergence shift (circuit - base) by model x band:
band         control  high   low  medium  very_high
model                                              
pythia-1.4b    -0.12 -0.22 -0.12    0.02      -0.11
pythia-160m     0.00  0.03  0.00   -0.01      -0.02
pythia-1b      -0.11 -0.06 -0.08   -0.16       0.02
pythia-410m    -0.13 -0.18  0.02   -0.04      -0.15
pythia-70m     -0.01 -0.01 -0.00   -0.00       0.01


In [14]:
# Attribution profile differences: base vs circuit
if not df_base_attrib.empty and not df_circuit_attrib.empty:
    # Overlay attention fraction trajectories
    for model in MODELS:
        fig, axes = plt.subplots(1, 2, figsize=(16, 6))

        # Left: attention fraction trajectory
        ax = axes[0]
        for band in BANDS:
            color = BAND_COLORS.get(band, "gray")
            label = BAND_NAMES.get(band, band)

            # Base
            base_bd = df_base_attrib[
                (df_base_attrib["model"] == model)
                & (df_base_attrib["band"] == band)
                & (df_base_attrib["draw"] == "draw_1")
            ].sort_values("layer")
            if not base_bd.empty:
                ax.plot(
                    base_bd["layer"],
                    base_bd["attn_frac"],
                    color=color,
                    linewidth=1.5,
                    marker="o",
                    markersize=2,
                    alpha=0.5,
                )

            # Circuit
            circ_bd = df_circuit_attrib[
                (df_circuit_attrib["model"] == model)
                & (df_circuit_attrib["band"] == band)
                & (df_circuit_attrib["draw"] == "draw_1")
            ].sort_values("layer")
            if not circ_bd.empty:
                ax.plot(
                    circ_bd["layer"],
                    circ_bd["attn_frac"],
                    color=color,
                    linewidth=2,
                    linestyle="--",
                    marker="s",
                    markersize=2,
                )

        # Manual legend entries
        ax.plot([], [], color="black", linewidth=1.5, alpha=0.5, label="Base")
        ax.plot([], [], color="black", linewidth=2, linestyle="--", label="Circuit")
        ax.set_xlabel("Layer")
        ax.set_ylabel("Attention Attribution Fraction")
        ax.set_title(f"Attention Fraction: Base vs Circuit")
        ax.legend(fontsize=8)
        ax.set_ylim(0, 1)

        # Right: delta attention fraction (circuit - base) per band
        ax = axes[1]
        for band in BANDS:
            color = BAND_COLORS.get(band, "gray")
            label = BAND_NAMES.get(band, band)

            base_bd = df_base_attrib[
                (df_base_attrib["model"] == model)
                & (df_base_attrib["band"] == band)
                & (df_base_attrib["draw"] == "draw_1")
            ].sort_values("layer")
            circ_bd = df_circuit_attrib[
                (df_circuit_attrib["model"] == model)
                & (df_circuit_attrib["band"] == band)
                & (df_circuit_attrib["draw"] == "draw_1")
            ].sort_values("layer")

            if not base_bd.empty and not circ_bd.empty:
                merged = pd.merge(
                    base_bd[["layer", "attn_frac"]].rename(
                        columns={"attn_frac": "base_attn_frac"}
                    ),
                    circ_bd[["layer", "attn_frac"]].rename(
                        columns={"attn_frac": "circ_attn_frac"}
                    ),
                    on="layer",
                )
                delta = merged["circ_attn_frac"] - merged["base_attn_frac"]
                ax.plot(
                    merged["layer"],
                    delta,
                    color=color,
                    label=label,
                    marker="o",
                    markersize=3,
                )

        ax.axhline(y=0, color="gray", linestyle="--", alpha=0.5)
        ax.set_xlabel("Layer")
        ax.set_ylabel("Delta Attention Fraction (Circuit - Base)")
        ax.set_title("Attribution Shift")
        ax.legend(fontsize=8)

        fig.suptitle(f"Attribution Comparison \u2014 {model}", y=1.02)
        fig.tight_layout()
        save_figure_comp(fig, f"viz_03c_08_attribution_comparison_{model}.png")

In [15]:
# Delta attribution heatmap: (circuit_attn_frac - base_attn_frac) by model x band
if not df_base_attrib.empty and not df_circuit_attrib.empty:
    base_summary = df_base_attrib.groupby(["model", "band"])["attn_frac"].mean()
    circ_summary = df_circuit_attrib.groupby(["model", "band"])["attn_frac"].mean()

    delta_df = (circ_summary - base_summary).reset_index()
    delta_df.columns = ["model", "band", "delta_attn_frac"]

    save_analysis_comp(delta_df, "03c_attribution_delta.csv")

    delta_wide = delta_df.pivot(index="model", columns="band", values="delta_attn_frac")
    delta_wide = delta_wide.reindex(
        index=MODELS, columns=[b for b in BANDS if b in delta_wide.columns]
    )

    vmax = max(abs(delta_wide.min().min()), abs(delta_wide.max().max()), 0.01)
    fig, ax = plt.subplots(figsize=(10, 5))
    sns.heatmap(
        delta_wide,
        annot=True,
        fmt=".3f",
        cmap="RdBu_r",
        square=True,
        linewidths=0,
        linecolor="none",
        vmin=-vmax,
        vmax=vmax,
        center=0,
        ax=ax,
    )
    ax.set_title("Delta Attention Fraction (Circuit - Base)")
    ax.set_xlabel("Band")
    ax.set_ylabel("Model")
    fig.tight_layout()
    save_figure_comp(fig, "viz_03c_09_attribution_delta_heatmap.png")

In [16]:
# Delta P(correct) at final layer: (circuit - base) by model x band
if not df_base_prob.empty and not df_circuit_prob.empty:

    def get_final_prob(df):
        """Get final-layer mean P(correct) per (model, band)."""
        return df.groupby(["model", "band"]).apply(
            lambda g: g.loc[g["layer"].idxmax(), "mean_prob_correct"]
        )

    base_final = get_final_prob(df_base_prob)
    circ_final = get_final_prob(df_circuit_prob)

    delta_prob_df = (circ_final - base_final).reset_index()
    delta_prob_df.columns = ["model", "band", "delta_final_prob"]

    save_analysis_comp(delta_prob_df, "03c_final_prob_delta.csv")

    delta_prob_wide = delta_prob_df.pivot(
        index="model", columns="band", values="delta_final_prob"
    )
    delta_prob_wide = delta_prob_wide.reindex(
        index=MODELS, columns=[b for b in BANDS if b in delta_prob_wide.columns]
    )

    vmax = max(abs(delta_prob_wide.min().min()), abs(delta_prob_wide.max().max()), 0.01)
    fig, ax = plt.subplots(figsize=(10, 5))
    sns.heatmap(
        delta_prob_wide,
        annot=True,
        fmt=".3f",
        cmap="RdBu_r",
        square=True,
        linewidths=0,
        linecolor="none",
        vmin=-vmax,
        vmax=vmax,
        center=0,
        ax=ax,
    )
    ax.set_title("Delta Final P(correct) (Circuit - Base)")
    ax.set_xlabel("Band")
    ax.set_ylabel("Model")
    fig.tight_layout()
    save_figure_comp(fig, "viz_03c_10_final_prob_delta_heatmap.png")

## 5. Cross-Model Patterns

Summarize circuit logit lens metrics across model scales and examine how
circuit ablation interacts with model size.

In [17]:
# Build master circuit logit lens DataFrame
rows_master = []

for model in MODELS:
    n_layers = MODEL_INFO[model]["n_layers"]
    for draw in DRAWS:
        for band in BANDS:
            record = {
                "model": model,
                "draw": draw,
                "band": band,
                "model_capacity": MODEL_CAPACITY[model],
                "n_layers": n_layers,
                "frequency_rank": FREQUENCY_RANK.get(band),
            }

            # Circuit convergence
            conv_row = (
                df_circuit_conv[
                    (df_circuit_conv["model"] == model)
                    & (df_circuit_conv["draw"] == draw)
                    & (df_circuit_conv["band"] == band)
                ]
                if not df_circuit_conv.empty
                else pd.DataFrame()
            )
            if len(conv_row) > 0:
                record["circuit_convergence_layer"] = conv_row.iloc[0][
                    "mean_convergence_layer"
                ]
                record["circuit_frac_convergence"] = conv_row.iloc[0][
                    "frac_convergence_layer"
                ]
                record["circuit_frac_converged"] = conv_row.iloc[0]["frac_converged"]

            # Circuit final P(correct)
            prob_rows = (
                df_circuit_prob[
                    (df_circuit_prob["model"] == model)
                    & (df_circuit_prob["draw"] == draw)
                    & (df_circuit_prob["band"] == band)
                ]
                if not df_circuit_prob.empty
                else pd.DataFrame()
            )
            if len(prob_rows) > 0:
                record["circuit_final_prob"] = prob_rows.loc[
                    prob_rows["layer"].idxmax(), "mean_prob_correct"
                ]

            # Circuit mean attribution
            attrib_rows = (
                df_circuit_attrib[
                    (df_circuit_attrib["model"] == model)
                    & (df_circuit_attrib["draw"] == draw)
                    & (df_circuit_attrib["band"] == band)
                ]
                if not df_circuit_attrib.empty
                else pd.DataFrame()
            )
            if len(attrib_rows) > 0:
                record["circuit_mean_attn_frac"] = attrib_rows["attn_frac"].mean()
                record["circuit_mean_mlp_frac"] = attrib_rows["mlp_frac"].mean()

            # Circuit summary (edge count)
            try:
                ps = load_prune_scores(model, band, draw)
                cs = get_circuit_summary(ps)
                record["circuit_edges"] = cs["total_edges"]
            except FileNotFoundError:
                pass

            rows_master.append(record)

df_master = pd.DataFrame(rows_master)
save_analysis_circuit(df_master, "03c_master_circuit_logit_lens.csv")
print(f"Master records: {len(df_master)}")

if not df_master.empty:
    summary_cols = [
        c
        for c in [
            "circuit_final_prob",
            "circuit_convergence_layer",
            "circuit_frac_convergence",
            "circuit_mean_attn_frac",
        ]
        if c in df_master.columns
    ]
    if summary_cols:
        print(f"\nKey circuit metrics by model (averaged across bands/draws):")
        print(df_master.groupby("model")[summary_cols].mean().round(3))

Master records: 75

Key circuit metrics by model (averaged across bands/draws):
             circuit_final_prob  circuit_convergence_layer  \
model                                                        
pythia-1.4b               0.523                     17.388   
pythia-160m               0.487                      9.015   
pythia-1b                 0.636                     11.515   
pythia-410m               0.540                     18.985   
pythia-70m                0.203                      4.508   

             circuit_frac_convergence  circuit_mean_attn_frac  
model                                                          
pythia-1.4b                     0.725                   0.550  
pythia-160m                     0.751                   0.401  
pythia-1b                       0.720                   0.597  
pythia-410m                     0.791                   0.527  
pythia-70m                      0.751                   0.494  


In [18]:
# Cross-model scaling: circuit metrics vs model size
if not df_master.empty:
    metrics_to_plot = [
        ("circuit_final_prob", "Circuit Final P(correct)"),
        ("circuit_frac_convergence", "Circuit Fractional Convergence"),
        ("circuit_mean_attn_frac", "Circuit Mean Attn Fraction"),
    ]
    metrics_to_plot = [(m, t) for m, t in metrics_to_plot if m in df_master.columns]

    if metrics_to_plot:
        n_panels = len(metrics_to_plot)
        fig, axes = plt.subplots(1, n_panels, figsize=(6 * n_panels, 5))
        if n_panels == 1:
            axes = [axes]

        for ax, (metric, title) in zip(axes, metrics_to_plot):
            for band in BANDS:
                band_data = df_master[df_master["band"] == band]
                if band_data.empty or metric not in band_data.columns:
                    continue
                means = band_data.groupby("model")[metric].mean()
                caps = band_data.groupby("model")["model_capacity"].first()
                color = BAND_COLORS.get(band, "gray")
                ax.plot(
                    caps,
                    means,
                    color=color,
                    label=BAND_NAMES.get(band, band),
                    marker="o",
                    markersize=5,
                )

            ax.set_xlabel("Model Capacity (M params)")
            ax.set_ylabel(title)
            ax.set_title(title)
            ax.set_xscale("log")

        axes[0].legend(fontsize=8)
        fig.suptitle("Circuit Logit Lens Metrics vs Model Size", y=1.02)
        fig.tight_layout()
        save_figure_circuit(fig, "viz_03c_11_cross_model_scaling.png")

In [19]:
# Overlay base vs circuit scaling for final P(correct)
if not df_base_prob.empty and not df_circuit_prob.empty:
    fig, ax = plt.subplots(figsize=(10, 6))

    for band in BANDS:
        color = BAND_COLORS.get(band, "gray")
        label = BAND_NAMES.get(band, band)

        # Base final P(correct) per model
        base_final = (
            df_base_prob.groupby(["model", "band"])
            .apply(lambda g: g.loc[g["layer"].idxmax(), "mean_prob_correct"])
            .reset_index()
        )
        base_final.columns = ["model", "band", "final_prob"]
        base_band = base_final[base_final["band"] == band]

        # Circuit final P(correct) per model
        circ_final = (
            df_circuit_prob.groupby(["model", "band"])
            .apply(lambda g: g.loc[g["layer"].idxmax(), "mean_prob_correct"])
            .reset_index()
        )
        circ_final.columns = ["model", "band", "final_prob"]
        circ_band = circ_final[circ_final["band"] == band]

        if not base_band.empty:
            caps = base_band["model"].map(MODEL_CAPACITY)
            ax.plot(
                caps,
                base_band["final_prob"],
                color=color,
                marker="o",
                markersize=5,
                linewidth=1.5,
            )

        if not circ_band.empty:
            caps = circ_band["model"].map(MODEL_CAPACITY)
            ax.plot(
                caps,
                circ_band["final_prob"],
                color=color,
                marker="s",
                markersize=5,
                linewidth=1.5,
                linestyle="--",
            )

    # Legend
    ax.plot([], [], color="black", marker="o", linewidth=1.5, label="Base")
    ax.plot(
        [],
        [],
        color="black",
        marker="s",
        linewidth=1.5,
        linestyle="--",
        label="Circuit",
    )
    ax.set_xlabel("Model Capacity (M params)")
    ax.set_ylabel("Final P(correct)")
    ax.set_title("Final P(correct) Scaling: Base vs Circuit")
    ax.set_xscale("log")
    ax.legend(fontsize=8)
    ax.set_ylim(0, 1)
    fig.tight_layout()
    save_figure_comp(fig, "viz_03c_12_prob_scaling_comparison.png")

In [20]:
# P(correct) vs fractional depth for circuit, overlaying all models (low-freq band)
if not df_circuit_prob.empty:
    fig, ax = plt.subplots(figsize=(12, 6))

    for model in MODELS:
        model_data = df_circuit_prob[
            (df_circuit_prob["model"] == model)
            & (df_circuit_prob["draw"] == "draw_1")
            & (df_circuit_prob["band"] == "low")
        ].sort_values("layer")
        if model_data.empty:
            continue

        n_layers = MODEL_INFO[model]["n_layers"]
        frac_depth = model_data["layer"].values / n_layers
        color = MODEL_COLORS.get(model, "gray")
        ax.plot(
            frac_depth,
            model_data["mean_prob_correct"],
            color=color,
            label=model,
            marker="o",
            markersize=3,
        )

    ax.set_xlabel("Fractional Depth (layer / n_layers)")
    ax.set_ylabel("Mean P(correct)")
    ax.set_title("Circuit P(correct) vs Fractional Depth \u2014 Low Frequency Band")
    ax.legend(fontsize=8)
    ax.set_ylim(bottom=0)
    fig.tight_layout()
    save_figure_circuit(fig, "viz_03c_13_circuit_prob_fractional_depth.png")

In [21]:
print("=" * 70)
print("NOTEBOOK 03c: LOGIT LENS & ATTRIBUTION \u2014 CIRCUIT ANALYSIS \u2014 SUMMARY")
print("=" * 70)

# 1. Circuit P(correct)
print("\n--- Circuit P(correct) Trajectory ---")
print(f"Total records: {len(df_circuit_prob)}")
if not df_circuit_prob.empty:
    for model in MODELS:
        m_data = df_circuit_prob[df_circuit_prob["model"] == model]
        if m_data.empty:
            continue
        final = m_data.groupby(["band"]).apply(
            lambda g: g.loc[g["layer"].idxmax(), "mean_prob_correct"]
        )
        print(f"  {model} final P(correct): {dict(final.round(3))}")

# 2. Convergence
print("\n--- Circuit Convergence ---")
print(f"Threshold: {CONVERGENCE_THRESHOLD}")
if not df_circuit_conv.empty:
    print(
        df_circuit_conv.groupby(["model", "band"])["mean_convergence_layer"]
        .mean()
        .unstack("band")
        .round(2)
    )

# 3. Attribution
print("\n--- Circuit Attribution ---")
if not df_circuit_attrib.empty:
    print(f"Method: {df_circuit_attrib['method'].unique()}")
    print(
        df_circuit_attrib.groupby(["model"])[["attn_frac", "mlp_frac"]].mean().round(3)
    )

# Output listing
print("\n" + "=" * 70)
print("NOTEBOOK 03c COMPLETE")
print("=" * 70)
print(f"\nCircuit CSVs in: {CIRCUIT_ANALYSIS}")
print(f"Circuit figures in: {CIRCUIT_VIZ}")
print(f"Comparison CSVs in: {COMP_ANALYSIS}")
print(f"Comparison figures in: {COMP_VIZ}")

for d, label in [
    (CIRCUIT_ANALYSIS, "Circuit analysis"),
    (CIRCUIT_VIZ, "Circuit viz"),
    (COMP_ANALYSIS, "Comparison analysis"),
    (COMP_VIZ, "Comparison viz"),
]:
    files = sorted(d.glob("*")) if d.exists() else []
    if files:
        print(f"\n{label}:")
        for f in files:
            print(f"  {f.name}")

NOTEBOOK 03c: LOGIT LENS & ATTRIBUTION: CIRCUIT ANALYSIS: SUMMARY

--- Circuit P(correct) Trajectory ---
Total records: 1230
  pythia-70m final P(correct): {'control': 0.236, 'high': 0.215, 'low': 0.1, 'medium': 0.169, 'very_high': 0.262}
  pythia-160m final P(correct): {'control': 0.519, 'high': 0.513, 'low': 0.393, 'medium': 0.457, 'very_high': 0.557}
  pythia-410m final P(correct): {'control': 0.489, 'high': 0.536, 'low': 0.579, 'medium': 0.557, 'very_high': 0.511}
  pythia-1b final P(correct): {'control': 0.573, 'high': 0.628, 'low': 0.658, 'medium': 0.714, 'very_high': 0.584}
  pythia-1.4b final P(correct): {'control': 0.553, 'high': 0.538, 'low': 0.495, 'medium': 0.482, 'very_high': 0.517}

--- Circuit Convergence ---
Threshold: 0.9
band         control   high    low  medium  very_high
model                                                
pythia-1.4b    15.03  16.57  21.45   19.39      14.50
pythia-160m     8.69   8.97   9.55    9.21       8.65
pythia-1b      11.05  11.35  12.40 